# 00 - Bronze: land the Velmora Excel files as Delta tables

Reads the ten `.xlsx` source files from the Lakehouse `Files` area and writes one Bronze Delta
table per file.

**Bronze rule applied here: every column is landed as a string.** Nothing is parsed, cleaned,
de-duplicated or type-cast. That is deliberate - the source files contain dates stored in four
different text formats, numbers that Excel typed inconsistently, and duplicate rows. Landing them
as text preserves exactly what the business sent, and gives the Silver notebook something real to
do. Three audit columns are added so every Silver row can be traced back to its source file and
Excel row.

### Before you run

1. Create a Lakehouse and **attach it to this notebook** (Explorer pane -> Add Lakehouse).
2. Upload the ten `.xlsx` files to `Files/bronze/` in that Lakehouse.
3. Run all cells.

## 1. Parameters

`LAKEHOUSE_ROOT` is where Fabric mounts the attached Lakehouse on the driver node. Leave it alone
unless you know you need something else.

In [ ]:
# PARAMETERS -----------------------------------------------------------------
LAKEHOUSE_ROOT = "/lakehouse/default"   # Fabric mounts the attached Lakehouse here
SOURCE_FOLDER  = "Files/bronze"         # folder holding the .xlsx files
BRONZE_PREFIX  = "brz_"                 # table name prefix for the Bronze layer
# -----------------------------------------------------------------------------

## 2. Imports and folder resolution

If `Files/bronze` does not exist, the notebook falls back to `Files` so it still works when the
files were dropped in the root.

In [ ]:
import os
import math
import datetime
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# Fabric-friendly write settings (harmless elsewhere)
for key, value in {
    "spark.sql.parquet.vorder.enabled": "true",
    "spark.microsoft.delta.optimizeWrite.enabled": "true",
}.items():
    try:
        spark.conf.set(key, value)
    except Exception as exc:                      # not a Fabric cluster - fine
        print(f"[info] could not set {key}: {exc}")

SOURCE_DIR = os.path.join(LAKEHOUSE_ROOT, SOURCE_FOLDER)
if not os.path.isdir(SOURCE_DIR):
    fallback = os.path.join(LAKEHOUSE_ROOT, "Files")
    print(f"[warn] {SOURCE_DIR} not found - falling back to {fallback}")
    SOURCE_DIR = fallback

print("Reading from:", SOURCE_DIR)
found = sorted(f for f in os.listdir(SOURCE_DIR) if f.lower().endswith(".xlsx"))
print(f"Found {len(found)} Excel file(s):")
for f_name in found:
    print("  -", f_name)

## 3. Source file to Bronze table map

Note the two sales files land in **separate** tables. They have different schemas, and reconciling
them is the schema-drift exercise handled in the Silver notebook - not here.

In [ ]:
FILES = {
    "Pharma_Product_Master.xlsx":            BRONZE_PREFIX + "pharma_products",
    "Pharma_Market_Master.xlsx":             BRONZE_PREFIX + "pharma_markets",
    "Pharma_Customer_Master.xlsx":           BRONZE_PREFIX + "pharma_customers",
    "Pharma_Sales_2025.xlsx":                BRONZE_PREFIX + "pharma_sales_2025",
    "Pharma_Sales_2026_Monthly_Update.xlsx": BRONZE_PREFIX + "pharma_sales_2026",
    "Pharma_Forecast_2026.xlsx":             BRONZE_PREFIX + "pharma_forecast",
    "Pharma_Budget_2026.xlsx":               BRONZE_PREFIX + "pharma_budget",
    "Pharma_Inventory.xlsx":                 BRONZE_PREFIX + "pharma_inventory",
    "Pharma_Operating_Expenses.xlsx":        BRONZE_PREFIX + "pharma_opex",
    "Pharma_FX_Rates.xlsx":                  BRONZE_PREFIX + "pharma_fx_rates",
}

for src, tgt in FILES.items():
    print(f"{src:42s} -> {tgt}")

## 4. Excel reader

`pandas.read_excel(dtype=object)` keeps every cell as Python read it, so a column holding both
real Excel dates and text dates is preserved rather than coerced.

`cell_to_string` then normalises each value to text:

- blanks (`NaN`, `NaT`, `None`) become a real `NULL`, never the string `"nan"`
- real Excel dates become `yyyy-MM-dd`
- whole numbers stored as floats (`2108.0`) lose the trailing `.0`
- text that was already a text date (`05/01/2025`) is left exactly as it was

That last point matters: the messy date formats survive into Bronze, which is what makes the
Silver date-parsing step a real exercise.

In [ ]:
def cell_to_string(value):
    """Normalise a single Excel cell to text, preserving source formatting quirks."""
    if value is None:
        return None
    if isinstance(value, float) and math.isnan(value):
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(value, (pd.Timestamp, datetime.datetime, datetime.date)):
        ts = pd.Timestamp(value)
        if (ts.hour, ts.minute, ts.second) == (0, 0, 0):
            return ts.strftime("%Y-%m-%d")
        return ts.strftime("%Y-%m-%d %H:%M:%S")

    if isinstance(value, bool):
        return "TRUE" if value else "FALSE"

    if isinstance(value, float) and float(value).is_integer():
        return str(int(value))

    text = str(value)
    return text if text.strip() != "" else None


def load_excel_as_strings(path):
    """Read one worksheet and return a pandas DataFrame where every column is text."""
    pdf = pd.read_excel(path, dtype=object, engine="openpyxl")
    pdf.columns = [str(c).strip() for c in pdf.columns]
    out = pd.DataFrame(index=pdf.index)
    for column in pdf.columns:
        out[column] = pdf[column].map(cell_to_string)
    return out


def null_safe(value):
    """Guard against pandas turning None back into NaN under its string dtype.

    pandas 3 infers a 'str' dtype whose missing marker is NaN, so a None returned by
    cell_to_string can reappear as a float. Landing that in a StringType column writes
    the literal text 'nan', which later looks like a valid ID and quietly defeats every
    null check in Silver. Normalise once, here.
    """
    if value is None:
        return None
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def to_spark(pdf, source_file):
    """Convert to a Spark DataFrame with an all-string schema plus audit columns."""
    schema = StructType([StructField(c, StringType(), True) for c in pdf.columns])
    rows = [tuple(null_safe(v) for v in r)
            for r in pdf.itertuples(index=False, name=None)]
    sdf = spark.createDataFrame(rows, schema=schema) if rows else spark.createDataFrame([], schema)
    return (sdf
            .withColumn("_SourceFile", F.lit(source_file))
            .withColumn("_SourceRowNumber", F.monotonically_increasing_id() + F.lit(2))
            .withColumn("_IngestedAtUTC", F.current_timestamp()))

print("Reader ready.")

## 5. Load every file

Existing Bronze tables are overwritten, so this notebook is safe to re-run.

In [ ]:
results = []

for file_name, table_name in FILES.items():
    path = os.path.join(SOURCE_DIR, file_name)
    if not os.path.exists(path):
        print(f"[SKIP] {file_name} not found in {SOURCE_DIR}")
        results.append((file_name, table_name, 0, 0, "MISSING"))
        continue

    pdf = load_excel_as_strings(path)
    sdf = to_spark(pdf, file_name)

    (sdf.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name))

    print(f"[OK]   {file_name:42s} -> {table_name:28s} {len(pdf):>6,} rows x {len(pdf.columns)} cols")
    results.append((file_name, table_name, len(pdf), len(pdf.columns), "LOADED"))

print("\nBronze load complete.")

## 6. Verify

The two sales tables should have **different column lists** - that is the schema drift you will
reconcile in the Silver notebook.

In [ ]:
summary = spark.createDataFrame(
    results, "SourceFile string, BronzeTable string, RowCount long, ColumnCount long, Status string")
summary.show(20, truncate=False)

print("Sales 2025 columns:")
print("  " + ", ".join(spark.table(BRONZE_PREFIX + "pharma_sales_2025").columns))
print("\nSales 2026 columns:")
print("  " + ", ".join(spark.table(BRONZE_PREFIX + "pharma_sales_2026").columns))

drift_2025 = set(spark.table(BRONZE_PREFIX + "pharma_sales_2025").columns)
drift_2026 = set(spark.table(BRONZE_PREFIX + "pharma_sales_2026").columns)
print("\nOnly in 2025:", sorted(drift_2025 - drift_2026))
print("Only in 2026:", sorted(drift_2026 - drift_2025))

In [ ]:
# Spot-check the messy date column: Bronze keeps all four text formats side by side.
(spark.table(BRONZE_PREFIX + "pharma_sales_2025")
      .select("TransactionID", "TransactionDate", "Currency", "ScenarioVersion")
      .filter("TransactionDate NOT RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}$'")
      .show(10, truncate=False))

## Next

Run `01_Silver_Cleanse_Standardise.ipynb`, which reads these `brz_*` tables and produces the
cleaned, typed, flagged `slv_*` tables.